# Workshop de Análise de Dados - Dia 2 (SOLUÇÃO DO INSTRUTOR)
## Análise e Comunicação

**Este notebook contém todas as respostas e comentários do instrutor. NÃO distribua antes do workshop.**

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandas numpy matplotlib seaborn
    !mkdir -p data/prepared
    BASE_URL = 'https://github.com/caiocgomes/workshop_dados/releases/download/v1/data'
    clean_files = ['orders_clean.csv', 'items_clean.csv', 'reviews_clean.csv']
    missing_files = [f for f in clean_files if not os.path.exists(f'data/prepared/{f}')]
    if missing_files:
        for f in clean_files:
            !wget -q -O data/prepared/{f} {BASE_URL}/{f}
    DATA_DIR = 'data/prepared'
    print('Ambiente Colab configurado!')
else:
    DATA_DIR = '../data/prepared'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

orders = pd.read_csv(f'{DATA_DIR}/orders_clean.csv', parse_dates=['purchase_date', 'approved_date', 'delivered_date', 'estimated_delivery_date'])
items = pd.read_csv(f'{DATA_DIR}/items_clean.csv')
reviews = pd.read_csv(f'{DATA_DIR}/reviews_clean.csv')

# Dataset consolidado
df = orders.merge(items, on='order_id', how='inner').merge(reviews, on='order_id', how='left')
df['total_item'] = df['price'] + df['freight_value']
df['delivery_days'] = (df['delivered_date'] - df['purchase_date']).dt.days
df['delay_days'] = (df['delivered_date'] - df['estimated_delivery_date']).dt.days
df['purchase_month'] = df['purchase_date'].dt.to_period('M')

print(f'Dataset consolidado: {len(df):,} linhas')

## 🎯 Demo do Instrutor: Margem de Erro (NÃO distribua)

Esta célula é exclusiva do notebook do instrutor. Projetar na tela durante a abertura do Dia 2, **antes** do Bloco 1. Mostra o ranking de categorias com barras de erro — o ranking colapsa para categorias com amostra pequena.

**Sequência:** groupby ao vivo → votação da sala → executar esta célula → "Vocês votaram num ranking que não existe."

In [ ]:
# === DEMO INSTRUTOR: NÃO DISTRIBUA ===
# Intervalo de confiança por categoria (margem de erro)
# Usar na abertura do Dia 2, antes do Bloco 1
import numpy as np

ci_data = (
    df[df['review_score'].notna()]
    .groupby('category')['review_score']
    .agg(['mean', 'std', 'count'])
    .rename(columns={'mean': 'nota_media', 'std': 'desvio', 'count': 'volume'})
)
ci_data['erro'] = 1.96 * ci_data['desvio'] / np.sqrt(ci_data['volume'])
ci_data['ic_lower'] = ci_data['nota_media'] - ci_data['erro']
ci_data['ic_upper'] = ci_data['nota_media'] + ci_data['erro']

# Selecionar: 6 piores + 6 melhores (volume >= 30 pra incluir os "perigosos")
ci_plot = ci_data[ci_data['volume'] >= 30].sort_values('nota_media')
plot_data = pd.concat([ci_plot.head(6), ci_plot.tail(6)]).drop_duplicates().sort_values('nota_media')

# Visualização
media_geral = df['review_score'].mean()
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if x < media_geral else '#2ecc71' for x in plot_data['nota_media']]
y_pos = range(len(plot_data))

ax.barh(y_pos, plot_data['nota_media'], color=colors, alpha=0.7, height=0.6)
ax.errorbar(plot_data['nota_media'], y_pos,
            xerr=plot_data['erro'],
            fmt='none', color='black', capsize=4, linewidth=1.5)

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{cat}  (n={int(v)})" for cat, v in
                     zip(plot_data.index, plot_data['volume'])])
ax.axvline(x=media_geral, color='gray', linestyle='--', alpha=0.5,
           label=f'Média geral: {media_geral:.2f}')
ax.set_xlabel('Nota Média')
ax.set_title('Satisfação por Categoria: Nota Média com Margem de Erro\n(Intervalo de confiança de 95%)')
ax.legend()
ax.set_xlim(1, 5)
plt.tight_layout()
plt.savefig('../materiais/viz_ic_demo.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Bloco 1: Análise Estatística

In [ ]:
# Análise 1: Satisfação por categoria
cat_stats = (
    df[df['review_score'].notna()]
    .groupby('category')
    .agg(
        nota_media=('review_score', 'mean'),
        nota_mediana=('review_score', 'median'),
        volume=('review_score', 'count'),
        receita=('price', 'sum'),
        pct_insatisfeitos=('review_score', lambda x: (x <= 2).mean()),
    )
    .round(3)
)

cat_stats_relevant = cat_stats[cat_stats['volume'] >= 100].sort_values('nota_media')

print('5 piores categorias:')
display(cat_stats_relevant.head(5))
print('\n5 melhores:')
display(cat_stats_relevant.tail(5))

# COMENTÁRIO INSTRUTOR: O spread entre melhor e pior é ~1 ponto (3.3 vs 4.3).
# O problema NÃO é generalizado - categorias como furniture_decor, office_furniture
# têm insatisfação estruturalmente maior. Provavelmente relacionado a logística
# (itens grandes, mais difíceis de entregar).

In [ ]:
# Análise 2: Atraso e satisfação
delivered = df[(df['order_status'] == 'delivered') & (df['review_score'].notna()) & (df['delay_days'].notna())].copy()

delivered['delay_bucket'] = pd.cut(
    delivered['delay_days'],
    bins=[-999, -7, 0, 7, 14, 999],
    labels=['Antecipado (>7d)', 'No prazo', 'Leve atraso (0-7d)', 'Atraso (7-14d)', 'Atraso grave (>14d)']
)

delay_analysis = delivered.groupby('delay_bucket', observed=True).agg(
    nota_media=('review_score', 'mean'),
    volume=('review_score', 'count'),
    pct_insatisfeitos=('review_score', lambda x: (x <= 2).mean()),
).round(3)

display(delay_analysis)

# COMENTÁRIO INSTRUTOR: O impacto é grande e monotônico.
# Antecipado: nota ~4.3. Atraso grave: nota ~2.0.
# A cada faixa de atraso, a nota cai ~0.5 pontos. Isso é evidência forte.

In [ ]:
# Frete vs satisfação
delivered['freight_quartile'] = pd.qcut(delivered['freight_value'], q=4, labels=['Q1 (barato)', 'Q2', 'Q3', 'Q4 (caro)'])

freight_analysis = delivered.groupby('freight_quartile', observed=True).agg(
    nota_media=('review_score', 'mean'),
    frete_medio=('freight_value', 'mean'),
    volume=('review_score', 'count')
).round(2)

display(freight_analysis)

corr = delivered[['delay_days', 'review_score', 'freight_value']].corr()
print(f'\nCorrelação atraso vs. nota: {corr.loc["delay_days", "review_score"]:.3f}')
print(f'Correlação frete vs. nota:  {corr.loc["freight_value", "review_score"]:.3f}')

# COMENTÁRIO INSTRUTOR: Atraso tem correlação muito mais forte com insatisfação
# do que frete. Isso refuta parcialmente a hipótese da equipe de logística:
# o problema não é o preço do frete, é o cumprimento do prazo.

In [ ]:
# Análise 3: Sazonalidade
monthly = (
    df[df['review_score'].notna()]
    .groupby('purchase_month')
    .agg(
        volume=('order_id', 'nunique'),
        nota_media=('review_score', 'mean'),
        receita=('price', 'sum'),
    )
    .round(2)
)

display(monthly)

# COMENTÁRIO INSTRUTOR: Há sazonalidade clara (nov/dez = pico).
# A nota média NÃO cai proporcionalmente ao volume, o que sugere
# que a operação escala razoavelmente. Mas vale investigar se
# atrasos aumentam nos picos.

In [ ]:
# Desafio: distribuição bimodal
top10_cats = df[df['review_score'].notna()].groupby('category').size().nlargest(10).index

for cat in top10_cats[:6]:
    scores = df[(df['category'] == cat) & (df['review_score'].notna())]['review_score']
    low = (scores <= 2).mean()
    high = (scores >= 4).mean()
    print(f'{cat:35s} | Insatisfeitos: {low:.1%} | Satisfeitos: {high:.1%} | Média: {scores.mean():.2f}')

---
## Bloco 2: Visualizações

In [ ]:
# Viz 1: Satisfação por categoria
fig, ax = plt.subplots(figsize=(12, 8))

worst5 = cat_stats_relevant.head(5)
best5 = cat_stats_relevant.tail(5)
plot_data = pd.concat([worst5, best5])['nota_media']

colors = ['#e74c3c'] * 5 + ['#2ecc71'] * 5
plot_data.plot(kind='barh', ax=ax, color=colors)

media_geral = df['review_score'].mean()
ax.set_title('Categorias com Melhor e Pior Satisfação\n(mínimo 100 avaliações)', fontsize=14)
ax.set_xlabel('Nota Média')
ax.axvline(x=media_geral, color='gray', linestyle='--', alpha=0.5, label=f'Média geral: {media_geral:.2f}')
ax.legend()

ax.annotate(
    'Categorias de móveis e itens grandes concentram a insatisfação, possivelmente por desafios logísticos.',
    xy=(0.5, -0.08), xycoords='axes fraction', fontsize=10, style='italic', color='gray', ha='center'
)

plt.tight_layout()
VIZ_DIR = f'{DATA_DIR}/../materiais' if not IN_COLAB else 'materiais'
os.makedirs(VIZ_DIR, exist_ok=True)
plt.savefig(f'{VIZ_DIR}/viz1_satisfacao_categoria.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Viz 2: Atraso vs satisfação
fig, ax = plt.subplots(figsize=(12, 6))

delay_plot = delay_analysis.reset_index()
bars = ax.bar(
    range(len(delay_plot)), delay_plot['nota_media'],
    color=['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c'],
    edgecolor='white'
)

for i, (bar, vol) in enumerate(zip(bars, delay_plot['volume'])):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'n={vol:,}', ha='center', fontsize=9)

ax.set_xticks(range(len(delay_plot)))
ax.set_xticklabels(delay_plot['delay_bucket'], rotation=15, ha='right')
ax.set_ylabel('Nota Média')
ax.set_title('Quanto Mais Atraso, Pior a Avaliação\nNota média por faixa de atraso na entrega', fontsize=14)
ax.set_ylim(0, 5.5)

ax.annotate(
    'Cada faixa de atraso reduz a nota em ~0.5 pontos. Entregas antecipadas têm as melhores avaliações.',
    xy=(0.5, -0.15), xycoords='axes fraction', fontsize=10, style='italic', color='gray', ha='center'
)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/viz2_atraso_satisfacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Viz 3: Evolução temporal
fig, ax1 = plt.subplots(figsize=(14, 6))

x = range(len(monthly))
ax1.bar(x, monthly['volume'], alpha=0.3, color='steelblue', label='Volume de pedidos')
ax1.set_ylabel('Volume de Pedidos', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot(x, monthly['nota_media'], color='coral', marker='o', linewidth=2, label='Nota média')
ax2.set_ylabel('Nota Média', color='coral')
ax2.tick_params(axis='y', labelcolor='coral')
ax2.set_ylim(1, 5)

ax1.set_xticks(x)
ax1.set_xticklabels([str(m) for m in monthly.index], rotation=45, ha='right')
ax1.set_title('Volume de Pedidos e Satisfação ao Longo do Tempo', fontsize=14)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

fig.text(0.5, -0.02, 'Volume cresce com sazonalidade clara, mas nota média permanece relativamente estável.',
         fontsize=10, style='italic', color='gray', ha='center')

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/viz3_evolucao_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Viz 4: Heatmap de satisfação - categoria vs faixa de atraso
top_cats = df[df['review_score'].notna()].groupby('category').size().nlargest(8).index
delivered_top = delivered[delivered['category'].isin(top_cats)]

pivot = delivered_top.pivot_table(
    values='review_score', 
    index='category', 
    columns='delay_bucket', 
    aggfunc='mean'
).round(2)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, annot=True, cmap='RdYlGn', vmin=1, vmax=5, ax=ax, fmt='.2f')
ax.set_title('Nota Média por Categoria e Faixa de Atraso\n(Top 8 categorias por volume)', fontsize=14)
ax.set_ylabel('Categoria')
ax.set_xlabel('Faixa de Atraso')

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/viz4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Bloco 3: Storytelling

## Documento de Análise: Diagnóstico de Satisfação do Marketplace

### Resumo Executivo

Analisamos ~99 mil pedidos do marketplace para diagnosticar os fatores por trás da queda reportada no NPS. A análise identificou que a insatisfação não é generalizada: está concentrada em categorias de itens grandes (móveis, decoração) e fortemente correlacionada com atraso na entrega. A cada faixa de atraso adicional, a nota média cai ~0.5 pontos, caindo de 4.3 (entregas antecipadas) para 2.0 (atrasos graves acima de 14 dias).

O valor do frete, isoladamente, tem impacto limitado na avaliação. Isso contradiz a hipótese inicial da equipe de logística. O problema principal não é quanto o cliente paga pelo frete, mas se o prazo prometido é cumprido.

O volume de vendas apresenta sazonalidade clara, com picos em novembro e dezembro. A nota média, no entanto, permanece relativamente estável mesmo nos períodos de pico, sugerindo que a operação escala adequadamente na maioria dos cenários.

Recomendamos priorizar a redução de atrasos nas categorias de itens grandes e revisar as estimativas de prazo de entrega para essas categorias.

### Contexto

A gerência de operações reportou queda no NPS do marketplace sem conseguir identificar a causa nos números agregados. Esta análise investiga três dimensões: satisfação por categoria de produto, relação entre logística e avaliação, e padrões temporais.

- **Período analisado:** Set/2016 a Ago/2018
- **Volume de pedidos:** ~99 mil pedidos únicos
- **Nota média geral:** ~4.0 (em 5)

### Descoberta 1: Insatisfação concentrada em categorias de itens grandes

As 5 categorias com pior satisfação são predominantemente de itens volumosos (furniture_decor, office_furniture, garden_tools), enquanto as melhores são de itens pequenos e fáceis de entregar. O spread é de ~1 ponto entre as melhores e piores categorias, o que é significativo numa escala de 1 a 5.

![Satisfação por Categoria](../materiais/viz1_satisfacao_categoria.png)

### Descoberta 2: Atraso na entrega é o principal driver de insatisfação

A relação entre atraso e nota é monotônica e forte: cada faixa de atraso adicional reduz a nota média em ~0.5 pontos. Entregas antecipadas recebem nota ~4.3, enquanto atrasos acima de 14 dias recebem nota ~2.0. O valor do frete tem correlação muito mais fraca com a nota.

![Impacto do Atraso](../materiais/viz2_atraso_satisfacao.png)

### Descoberta 3: Volume sazonal, mas satisfação estável

O volume de vendas tem sazonalidade esperada (picos em novembro/dezembro), mas a nota média não deteriora proporcionalmente nos períodos de pico. Isso sugere que a operação escala razoavelmente, e que o problema de insatisfação é estrutural (certas categorias e rotas de entrega), não sazonal.

![Evolução Temporal](../materiais/viz3_evolucao_temporal.png)

### Recomendações

1. **Priorizar redução de atrasos em categorias volumosas:** Renegociar SLAs com transportadoras ou segmentar rotas de entrega para itens grandes. O impacto potencial é de ~0.5 pontos na nota média dessas categorias.

2. **Revisar estimativas de prazo:** Prometer um prazo mais longo e entregar antes tem efeito positivo maior do que prometer prazo curto e atrasar. Considerar adicionar buffer nas estimativas de categorias de risco.

### Limitações e Próximos Passos

Esta análise identifica correlações, não causalidade. Para quantificar o impacto de reduzir atrasos em X dias sobre a satisfação, seria necessário um experimento controlado ou análise causal com variáveis instrumentais. Também não investigamos o efeito de região geográfica, que pode ser confundidor entre categoria e atraso.

In [ ]:
# Exportação
import subprocess

if IN_COLAB:
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', '--no-input',
         '--output', 'analise_marketplace',
         '/content/solucao_dia2.ipynb'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        from google.colab import files
        print('Documento exportado! Baixando...')
        files.download('analise_marketplace.html')
    else:
        print(f'Erro: {result.stderr}')
else:
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', '--no-input',
         '--output', '../materiais/analise_marketplace',
         'solucao_dia2.ipynb'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('Documento exportado: ../materiais/analise_marketplace.html')
    else:
        print(f'Erro: {result.stderr}')